# Da'wah Bot Analysis Workbook Builder

This notebook reproduces the four analysis workbooks for the Islamic Da'wah bot conversation dataset:

1. `islamic_dawah_topics_analysis.xlsx`
2. `existing_muslim_dawah_bot_analysis_no_tokens.xlsx`
3. `objection_blocker_barrier_analysis.xlsx`
4. `religion_specific_concern_playbooks.xlsx`

The notebook uses the existing analysis scripts in this project so the taxonomy rules, grouping logic, and workbook formatting stay consistent with the delivered files.

**Important:** token columns are not used. The notebook checks the source CSV and stops if any token column is present.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import textwrap

import pandas as pd
try:
    from IPython.display import HTML, display
except ImportError:
    class HTML(str):
        pass

    def display(value):
        print(value)

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 120)

ROOT = Path.cwd()
if not (ROOT / "outputs").exists():
    # Fallback for opening the notebook from another working directory.
    ROOT = Path(r"D:\AI Projects\Islam Chat analysis (April)")

SOURCE_CSV = (
    ROOT
    / "New task conversation analysis (April)"
    / "LLM point extraction and analysis"
    / "full_conversation_points_extraction"
    / "full_conversation_points_extraction.csv"
)

ANALYSIS_STEPS = [
    {
        "name": "Topic analysis",
        "folder": ROOT / "outputs" / "topic_analysis",
        "python_script": ROOT / "outputs" / "topic_analysis" / "aggregate_topics.py",
        "workbook_script": ROOT / "outputs" / "topic_analysis" / "build_topic_workbook.mjs",
        "json_output": ROOT / "outputs" / "topic_analysis" / "topic_analysis_data.json",
        "xlsx_output": ROOT / "outputs" / "topic_analysis" / "islamic_dawah_topics_analysis.xlsx",
    },
    {
        "name": "Existing-Muslim analysis, no tokens",
        "folder": ROOT / "outputs" / "existing_muslim_analysis",
        "python_script": ROOT / "outputs" / "existing_muslim_analysis" / "aggregate_existing_muslim.py",
        "workbook_script": ROOT / "outputs" / "existing_muslim_analysis" / "build_existing_muslim_workbook.mjs",
        "json_output": ROOT / "outputs" / "existing_muslim_analysis" / "existing_muslim_analysis_data.json",
        "xlsx_output": ROOT / "outputs" / "existing_muslim_analysis" / "existing_muslim_dawah_bot_analysis_no_tokens.xlsx",
    },
    {
        "name": "Objection and blocker barrier analysis",
        "folder": ROOT / "outputs" / "barrier_analysis",
        "python_script": ROOT / "outputs" / "barrier_analysis" / "aggregate_barriers.py",
        "workbook_script": ROOT / "outputs" / "barrier_analysis" / "build_barrier_workbook.mjs",
        "json_output": ROOT / "outputs" / "barrier_analysis" / "barrier_analysis_data.json",
        "xlsx_output": ROOT / "outputs" / "barrier_analysis" / "objection_blocker_barrier_analysis.xlsx",
    },
    {
        "name": "Religion-specific concern playbooks",
        "folder": ROOT / "outputs" / "religion_playbooks",
        "python_script": ROOT / "outputs" / "religion_playbooks" / "aggregate_religion_playbooks.py",
        "workbook_script": ROOT / "outputs" / "religion_playbooks" / "build_religion_playbook_workbook.mjs",
        "json_output": ROOT / "outputs" / "religion_playbooks" / "religion_playbook_data.json",
        "xlsx_output": ROOT / "outputs" / "religion_playbooks" / "religion_specific_concern_playbooks.xlsx",
    },
]

print("Project root:", ROOT)
print("Source CSV:", SOURCE_CSV)


## 1. Validate The Source Data

This cell checks that the expected CSV exists and confirms the notebook is not using token columns.


In [ ]:
if not SOURCE_CSV.exists():
    raise FileNotFoundError(f"Source CSV not found: {SOURCE_CSV}")

header = pd.read_csv(SOURCE_CSV, nrows=0).columns.tolist()
token_columns = [col for col in header if "token" in col.lower()]
if token_columns:
    raise ValueError(
        "Token columns are present in the source CSV. Remove them before running this notebook: "
        + ", ".join(token_columns)
    )

required_columns = {
    "general_chat_id",
    "conversation_summary",
    "suspected_religion",
    "is_existing_muslim",
    "user_language",
    "conversation_type",
    "engagement_score",
    "engagement_flow",
    "topics_discussed",
    "key_blocker",
    "user_objections",
    "start_mood",
    "end_mood",
    "user_intent",
    "conversion_funnel",
    "script_dumping",
    "response_quality",
    "bot_critique",
}
missing = sorted(required_columns - set(header))
if missing:
    raise ValueError("Missing required columns: " + ", ".join(missing))

row_count = len(pd.read_csv(SOURCE_CSV, usecols=["general_chat_id"]))
print(f"Conversation rows in source CSV: {row_count:,}")
print(f"Columns available: {len(header)}")
print("Token columns found: none")


## 2. Rebuild The Analysis Outputs

Each analysis has two stages:

- Python aggregation script: reads the CSV and writes a structured `.json` summary.
- Workbook builder script: reads the JSON and writes the formatted `.xlsx` workbook.

The workbook builders use Node.js because the original workbooks were created with JavaScript spreadsheet tooling.


In [ ]:
def find_node():
    candidates = []
    if os.environ.get("NODE_EXE"):
        candidates.append(Path(os.environ["NODE_EXE"]))
    bundled = (
        Path.home()
        / ".cache"
        / "codex-runtimes"
        / "codex-primary-runtime"
        / "dependencies"
        / "node"
        / "bin"
        / ("node.exe" if os.name == "nt" else "node")
    )
    candidates.append(bundled)
    if shutil.which("node"):
        candidates.append(Path(shutil.which("node")))
    for candidate in candidates:
        if candidate and candidate.exists():
            return str(candidate)
    raise FileNotFoundError("Node.js was not found. Install Node.js or set NODE_EXE to the node executable.")


NODE_EXE = find_node()
print("Python:", sys.executable)
print("Node:", NODE_EXE)


def run_command(command, cwd, expected_output=None):
    expected_output = Path(expected_output) if expected_output else None
    before_mtime = expected_output.stat().st_mtime_ns if expected_output and expected_output.exists() else None
    result = subprocess.run(
        [str(part) for part in command],
        cwd=str(cwd),
        text=True,
        capture_output=True,
        check=False,
    )
    if result.stdout:
        print(result.stdout.strip())
    if result.stderr:
        print(result.stderr.strip())
    if result.returncode != 0:
        if expected_output and expected_output.exists():
            after_mtime = expected_output.stat().st_mtime_ns
            if before_mtime is None or after_mtime >= before_mtime:
                print(
                    f"Warning: command returned exit code {result.returncode}, "
                    f"but expected output exists: {expected_output}"
                )
                return result
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {command}")
    return result


In [ ]:
for step in ANALYSIS_STEPS:
    print("\n" + "=" * 90)
    print(step["name"])
    print("=" * 90)
    run_command([sys.executable, step["python_script"]], cwd=ROOT)
    run_command([NODE_EXE, step["workbook_script"]], cwd=step["folder"], expected_output=step["xlsx_output"])

print("\nAll analysis workbooks rebuilt.")


## 3. Validate Generated Files

This confirms that each JSON summary and final Excel workbook exists after the pipeline runs.


In [ ]:
validation_rows = []
for step in ANALYSIS_STEPS:
    for output_type in ["json_output", "xlsx_output"]:
        path = step[output_type]
        validation_rows.append(
            {
                "analysis": step["name"],
                "output_type": output_type.replace("_output", ""),
                "exists": path.exists(),
                "size_kb": round(path.stat().st_size / 1024, 1) if path.exists() else None,
                "path": str(path.relative_to(ROOT)) if path.exists() else str(path),
            }
        )
validation_df = pd.DataFrame(validation_rows)
display(validation_df)
if not validation_df["exists"].all():
    raise FileNotFoundError("One or more expected analysis outputs were not created.")


## 4. Helper Functions For Notebook Summaries

The next cells load the generated JSON files and show the same high-level findings used in the workbooks.


In [ ]:
def load_step_json(step_name):
    step = next(item for item in ANALYSIS_STEPS if item["name"] == step_name)
    return json.loads(step["json_output"].read_text(encoding="utf-8"))


def metrics_frame(data):
    return pd.DataFrame(
        [{"metric": key, "value": value} for key, value in data["metrics"].items()]
    )


def show_bar(df, label_col, value_col, title, color="#176d89", max_rows=12):
    plot_df = df.head(max_rows).copy()
    max_value = plot_df[value_col].max() or 1
    rows = []
    for _, row in plot_df.iterrows():
        width = max(2, float(row[value_col]) / max_value * 100)
        rows.append(
            f"""
            <div style='display:grid;grid-template-columns:280px 1fr 90px;gap:12px;align-items:center;margin:7px 0;'>
              <div style='font-weight:600'>{row[label_col]}</div>
              <div style='background:#e8eef2;border-radius:999px;overflow:hidden;height:18px;'>
                <div style='width:{width:.1f}%;background:{color};height:18px;'></div>
              </div>
              <div style='font-weight:700;text-align:right'>{int(row[value_col]):,}</div>
            </div>
            """
        )
    display(
        HTML(
            f"""
            <div style='border:1px solid #dfe9e5;border-radius:8px;padding:14px;margin:8px 0;background:white'>
              <h3 style='margin:0 0 8px'>{title}</h3>
              {''.join(rows)}
            </div>
            """
        )
    )


## 5. Topic Analysis

This analysis identifies the most common themes in `topics_discussed`.


In [ ]:
topic_data = load_step_json("Topic analysis")
display(metrics_frame(topic_data))

topic_theme_df = pd.DataFrame(topic_data["theme_rows"])
topic_theme_no_long_tail = topic_theme_df[topic_theme_df["theme"] != "Other / Long-tail Topics"].copy()
display(topic_theme_no_long_tail[["theme", "mentions", "conversations", "share_of_topic_conversations", "example_exact_topics"]].head(15))
show_bar(topic_theme_no_long_tail, "theme", "mentions", "Top topic themes by mentions")


## 6. Existing-Muslim Traffic Analysis

This analysis profiles users where `is_existing_muslim == True`. It intentionally excludes token metrics.


In [ ]:
muslim_data = load_step_json("Existing-Muslim analysis, no tokens")
display(metrics_frame(muslim_data))

muslim_type_df = pd.DataFrame(muslim_data["muslim_conversation_type_rows"])
muslim_need_df = pd.DataFrame(muslim_data["muslim_ask_cluster_rows"])
redirect_df = pd.DataFrame(muslim_data["muslim_redirect_quality_rows"])

display(muslim_type_df)
show_bar(muslim_type_df, "conversation_type", "conversations", "Conversation type within existing-Muslim users", color="#5c8a4a")

display(muslim_need_df)
show_bar(muslim_need_df, "ask_cluster", "conversations", "What existing Muslims ask for", color="#5c8a4a")

display(redirect_df)


## 7. Objection And Blocker Mapping

This analysis combines `user_objections` and `key_blocker` into a barrier taxonomy.


In [ ]:
barrier_data = load_step_json("Objection and blocker barrier analysis")
display(metrics_frame(barrier_data))

barrier_df = pd.DataFrame(barrier_data["barrier_rows"])
actionable_barriers = barrier_df[barrier_df["barrier"] != "Other / Unclustered Barrier"].copy()
display(actionable_barriers[["barrier", "barrier_type", "conversations", "mentions", "content_need"]].head(12))
show_bar(actionable_barriers, "barrier", "conversations", "Top actionable barriers", color="#d95f59")

matrix_df = pd.DataFrame(barrier_data["barrier_religion_matrix"])
display(matrix_df.head(15))


## 8. Religion-Specific Concern Playbooks

This analysis creates one profile per religion segment using normalized topics, objections, key blockers, and mood arcs.


In [ ]:
religion_data = load_step_json("Religion-specific concern playbooks")
display(metrics_frame(religion_data))

profile_df = pd.DataFrame(religion_data["profiles"])
profile_cols = [
    "religion",
    "conversations",
    "share_of_all_conversations",
    "dominant_blocker",
    "typical_mood_arc",
    "top_topics_summary",
    "top_objections_summary",
    "script_emphasis",
]
display(profile_df[profile_cols])
show_bar(profile_df, "religion", "conversations", "Conversation volume by suspected religion")

topic_matrix_df = pd.DataFrame(religion_data["topic_matrix_rows"])
barrier_matrix_df = pd.DataFrame(religion_data["barrier_matrix_rows"])
display(topic_matrix_df.head(12))
display(barrier_matrix_df.head(12))


## 9. Final Workbook Outputs

These are the final files created by the notebook.


In [ ]:
final_files = pd.DataFrame(
    [
        {
            "analysis": step["name"],
            "workbook": str(step["xlsx_output"].relative_to(ROOT)),
            "size_kb": round(step["xlsx_output"].stat().st_size / 1024, 1),
        }
        for step in ANALYSIS_STEPS
    ]
)
display(final_files)
